# Notebook 3 — Models & Purged Walk-Forward

**This notebook's job:** produce honest out-of-sample variance forecasts from every model, at both horizons. The order of operations is enforced:

1. Tune the ML models on the **validation folds** (2019–2022) only.
2. **Freeze** every choice in `frozen_config.yaml`, with a hash.
3. Generate **test** forecasts (2023–2025) with the frozen configuration, refitting once a year as a live desk would.

**Scoring the test set is deliberately left to Notebook 4.** This notebook never computes a test-period loss.

| | |
|---|---|
| **Inputs** | `features.parquet`, `feature_sets.json`, `panel.parquet`, `targets.parquet` |
| **Outputs** | `oos_forecasts.parquet` · `frozen_config.yaml` · `optuna_trials.csv` · `regimes.parquet` · `garch_params.parquet` |
| **Runtime** | about 20–40 min on a laptop, mostly Optuna (30 trials × 2 models × 2 horizons) |

**Model hierarchy.** Every forecast is annualised variance over *t+1…t+h*.

| Level | Models | Inputs | How fitted |
|---|---|---|---|
| 1 Naive | HistVol (random walk), EWMA λ=0.94 | returns | no fitting |
| 2 Econometric | GARCH, EGARCH, GJR (all Student-t) | returns | refit **monthly**, expanding window |
| 2 Econometric | HAR, HAR-X (+ log VIX²) | 3–4 fixed features | OLS on log RV, refit per fold |
| 3 ML | Random Forest, XGBoost | 61 features | Optuna on validation QLIKE, refit per fold |
| 4 Uncertainty | Gradient-boosted quantiles (10/50/90%) | 61 features | fixed settings, refit per fold |
| Ablations | XGB on E1, E2, E3, no-VIX; RF no-VIX | feature subsets | tuned XGB/RF settings reused |

**Why HAR-X is the headline benchmark.** Notebook 2 found that VIX alone has a 0.70 rank correlation with future variance. Beating GARCH, which sees only past returns, says little. Beating HAR-X answers H1 properly: does ML add anything beyond a linear model that already knows implied volatility?

## 0. Setup

In [ ]:
import os, sys, json, time, platform, hashlib
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Open this notebook from inside the repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # harmless tqdm notice from optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import arch, sklearn, xgboost, optuna
from statsmodels.stats.diagnostic import acorr_ljungbox

from src.config import load_config
from src.data import acquire
from src import models as M
from src.metrics import qlike, loss_table
from src.pipeline import forecast_folds, regimes_for_folds
from src.validation import make_folds, check_fold

cfg, P = load_config(ROOT)
SEED = cfg["project"]["seed"]
H = cfg["targets"]["horizons"]
MC = cfg["models"]
DATA_MODE = os.environ.get("VOLRES_DATA_MODE", "live")
N_TRIALS = int(os.environ.get("VOLRES_TRIALS", MC["optuna_trials"]))   # env override only for quick pipeline tests
sub = "_synthetic" if DATA_MODE == "synthetic" else ""
OUT = P.processed / sub if sub else P.processed
FIG = P.figures / sub if sub else P.figures
RES = P.results / sub if sub else P.results
for p in (FIG, RES):
    p.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4g}")
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

def savefig(fig, name):
    fig.savefig(FIG / f"nb03_{name}.png", bbox_inches="tight", dpi=150)

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:5.1f} min] {msg}")

if DATA_MODE == "synthetic":
    print("=" * 72 + "\nSYNTHETIC MODE: pipeline test only. These numbers are NOT research results.\n" + "=" * 72)
env = {"python": platform.python_version(), "pandas": pd.__version__, "arch": arch.__version__,
       "sklearn": sklearn.__version__, "xgboost": xgboost.__version__, "optuna": optuna.__version__,
       "data_mode": DATA_MODE, "optuna_trials": N_TRIALS}
print(env)

## 1. Load inputs and verify provenance

In [ ]:
meta1 = json.loads((OUT / "nb01_meta.json").read_text())
meta2 = json.loads((OUT / "nb02_meta.json").read_text())
for name, meta in [("panel.parquet", meta1), ("targets.parquet", meta1),
                   ("features.parquet", meta2), ("feature_sets.json", meta2)]:
    assert acquire.sha256(OUT / name) == meta["outputs_sha256"][name], f"{name} changed since it was produced"

panel = pd.read_parquet(OUT / "panel.parquet")
targets = pd.read_parquet(OUT / "targets.parquet")
X = pd.read_parquet(OUT / "features.parquet")
feature_sets = json.loads((OUT / "feature_sets.json").read_text())
assert panel.index.equals(X.index) and panel.index.equals(targets.index)
print(f"features {X.shape}, feature sets: {list(feature_sets)}")

## 2. Folds

- **Validation folds (2019–2022):** expanding training window from Jan 2015, evaluated on one year each.
- **Test folds (2023–2025):** the same construction. Models are refit once a year using only data before that year.

For every training row, the target window (*t+1…t+h*) must end before the evaluation year starts. Rows whose labels cross into it are **purged**: 5 rows at *h* = 5 and 20 rows at *h* = 20 per boundary. `check_fold` asserts this for every fold.

In [ ]:
s0 = cfg["data"]["sample_start"]; s1 = cfg["data"]["sample_end"]
val_folds = {h: make_folds(panel.index, targets[f"label_end_{h}"], s0, cfg["splits"]["val_years"], "val", s1) for h in H}
test_folds = {h: make_folds(panel.index, targets[f"label_end_{h}"], s0, MC["test_years"], "test", s1) for h in H}
rows = []
for h in H:
    for f in val_folds[h] + test_folds[h]:
        check_fold(f, targets[f"label_end_{h}"])
        rows.append({"h": h, "fold": f.name, "train": f"{f.train.min().date()} .. {f.train.max().date()}",
                     "n_train": len(f.train), "purged": f.n_purged,
                     "evaluate": f"{f.evaluate.min().date()} .. {f.evaluate.max().date()}", "n_eval": len(f.evaluate)})
pd.DataFrame(rows).set_index(["h", "fold"])

## 3. GARCH family on the development period (diagnostics)

Before the walk-forward, each GARCH specification is fit once on 2015–2022 to look at the estimates. This is descriptive only. The walk-forward refits monthly using only data before each month.

Three questions:
- **Persistence:** α + β (+ γ/2 for GJR). Close to 1 means shocks decay slowly. The half-life of a shock is ln 0.5 / ln(persistence).
- **Asymmetry:** is γ significantly positive in GJR, or negative in EGARCH? This is the leverage effect from Notebook 1.
- **Fit adequacy:** are the standardised residuals *z* and *z*² white noise? This is the correct test for return predictability that Notebook 1 deferred. The raw Ljung-Box test over-rejects under volatility clustering.

In [ ]:
from arch import arch_model
dev_ret = panel["ret"].loc[s0:"2022-12-31"].dropna() * 100
diag, fits = [], {}
for name, spec in M.GARCH_SPECS.items():
    kw = {k: v for k, v in spec.items() if k != "method"}
    res = arch_model(dev_ret, mean="Constant", dist="t", **kw).fit(disp="off")
    fits[name] = res
    p = res.params
    pers = p["alpha[1]"] + p["beta[1]"] + (p.get("gamma[1]", 0) / 2 if name == "GJR" else 0)
    if name == "EGARCH":
        pers = p["beta[1]"]
    z = res.std_resid.dropna()
    lb_z = acorr_ljungbox(z, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
    lb_z2 = acorr_ljungbox(z ** 2, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
    diag.append({"model": name, "persistence": pers, "half_life_days": np.log(0.5) / np.log(pers),
                 "gamma": p.get("gamma[1]", np.nan), "gamma_tstat": res.tvalues.get("gamma[1]", np.nan),
                 "nu (t dof)": p["nu"], "loglik": res.loglikelihood, "BIC": res.bic,
                 "LB(10) z p": lb_z, "LB(10) z² p": lb_z2})
diag = pd.DataFrame(diag).set_index("model")
diag.to_csv(RES / "garch_dev_diagnostics.csv")
diag

**Reading it.**
- ν well below 30 confirms fat tails; Student-t was the right choice.
- A large t-statistic on γ confirms the leverage effect.
- The lowest BIC is the best in-sample fit. Whether that carries over out of sample is a separate question, answered in Notebook 4.
- If LB on *z*² is no longer significant, the GARCH model has absorbed the clustering.
- Residual autocorrelation in *z* is typically much weaker than in raw returns. That confirms the Notebook 1 Ljung-Box rejection was mostly heteroskedasticity, not predictable returns.

## 4. Tune the ML models (validation folds only)

The objective is the mean validation **QLIKE** across the four folds, computed on variance after the log-to-level correction, so the tuning loss is the same as the evaluation loss.

The budget is 30 TPE trials per model per horizon, fixed in `config.yaml` before tuning. **Every trial is logged**, and the total trial count feeds the Deflated Sharpe Ratio in Notebook 5.

In [ ]:
studies, trial_rows = {}, []
for h in H:
    y_log, rv = targets[f"rv_logvar_{h}"], targets[f"rv_var_{h}"]
    Xp = X[feature_sets[MC["primary_feature_set"]]]
    for model in ("XGB", "RF"):
        t = time.time()
        st = M.tune(model, Xp, y_log, rv, val_folds[h], targets[f"label_end_{h}"], N_TRIALS, SEED)
        studies[f"{model}_h{h}"] = st
        df = st.trials_dataframe()
        df.insert(0, "study", f"{model}_h{h}")
        trial_rows.append(df)
        log(f"{model} h={h}: best val QLIKE {st.best_value:.4f} after {len(st.trials)} trials ({time.time() - t:.0f}s)")

trials = pd.concat(trial_rows, ignore_index=True)
trials.to_csv(RES / "optuna_trials.csv", index=False)
best_params = {k: st.best_params for k, st in studies.items()}
pd.DataFrame(best_params).T

In [ ]:
fig, axes = plt.subplots(1, len(studies), figsize=(4 * len(studies), 3.3), sharey=False)
for ax, (k, st) in zip(axes, studies.items()):
    v = np.array([t.value for t in st.trials])
    ax.plot(v, "o", ms=3, alpha=0.5, label="trial")
    ax.plot(np.minimum.accumulate(v), "-", color="tab:red", label="best so far")
    ax.set_title(k, fontsize=10); ax.set_xlabel("trial")
axes[0].set_ylabel("mean validation QLIKE"); axes[0].legend(fontsize=8)
fig.suptitle("Optuna search: a flattening red line means the 30-trial budget was enough", fontsize=10)
fig.tight_layout(); savefig(fig, "optuna_history"); plt.show()

## 5. Freeze the configuration

Everything that defines a forecast is written to `frozen_config.yaml`: tuned hyper-parameters, feature set, refit schedules, seeds and input hashes. The file's own hash is recorded.

**From here on nothing may change.** If you edit anything above and rerun, the hash changes, and Notebook 4 records which configuration produced the test results.

In [ ]:
frozen = {
    "frozen_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    "seed": SEED,
    "horizons": H,
    "primary_set": MC["primary_feature_set"],
    "best_params": best_params,
    "ewma_lambda": MC["ewma_lambda"],
    "garch_fit_start": s0,
    "garch_refit": "monthly, expanding window, Student-t, constant mean",
    "egarch_simulations": MC["egarch_simulations"],
    "calib_rows": MC["calib_rows"],
    "log_to_level": "exp(var(e)/2), e = residuals on last calib_rows of each training window",
    "quantiles": MC["quantiles"],
    "optuna_trials_per_study": N_TRIALS,
    "total_trials": int(len(trials)),
    "val_years": cfg["splits"]["val_years"],
    "test_years": MC["test_years"],
    "inputs_sha256": {"features.parquet": meta2["outputs_sha256"]["features.parquet"],
                      "targets.parquet": meta1["outputs_sha256"]["targets.parquet"]},
}
text = yaml.safe_dump(frozen, sort_keys=False)
(OUT / "frozen_config.yaml").write_text(text)
FROZEN_HASH = hashlib.sha256(text.encode()).hexdigest()
print(text)
print("frozen_config sha256:", FROZEN_HASH)

## 6. Validation-period forecasts

All models are run on the four validation folds with the frozen configuration.

The scorecard below is **optimistic for RF and XGB**, because their hyper-parameters were chosen on these same folds. Treat it as a sanity check that nothing is broken, not as the result. The honest comparison is the test set in Notebook 4.

In [ ]:
fc_val, gp_val = forecast_folds(val_folds, X, targets, panel, feature_sets, frozen, log=log)
log(f"validation forecasts: {len(fc_val):,} rows, {fc_val.model.nunique()} models")

def scorecard(fc):
    rows = []
    for (h, m), g in fc.groupby(["horizon", "model"]):
        rv = targets.loc[g.date, f"rv_var_{h}"].to_numpy()
        rows.append({"horizon": h, "model": m, "QLIKE": float(np.mean(qlike(rv, g.forecast))),
                     "RMSE_vol": float(np.sqrt(np.mean((np.sqrt(rv) - np.sqrt(g.forecast)) ** 2))),
                     "bias_vol": float(np.mean(np.sqrt(g.forecast) - np.sqrt(rv)))})
    return pd.DataFrame(rows)

point_models = [m for m in fc_val.model.unique() if not m.startswith("QGB_")]
sc_val = scorecard(fc_val[fc_val.model.isin(point_models)])
board = sc_val.pivot(index="model", columns="horizon", values="QLIKE")
board.columns = [f"QLIKE h={h}" for h in board.columns]
board = board.sort_values(board.columns[-1])
sc_val.to_csv(RES / "validation_scorecard.csv", index=False)
board

In [ ]:
core = ["HistVol", "EWMA", "GARCH", "EGARCH", "GJR", "HAR", "HAR_X", "RF", "XGB"]
fig, axes = plt.subplots(1, len(H), figsize=(6 * len(H), 3.8), sharey=False)
for ax, h in zip(np.atleast_1d(axes), H):
    s = sc_val[(sc_val.horizon == h) & sc_val.model.isin(core)].set_index("model").loc[core, "QLIKE"]
    colors = ["#999"] * 2 + ["tab:blue"] * 3 + ["tab:green"] * 2 + ["tab:red"] * 2
    ax.bar(s.index, s.values, color=colors)
    ax.axhline(s["HAR_X"], color="tab:green", ls="--", lw=1)
    ax.set_title(f"Validation QLIKE, h={h} (lower is better)"); ax.tick_params(axis="x", rotation=45)
fig.tight_layout(); savefig(fig, "validation_qlike"); plt.show()

### 6.1 How the forecasts behave through the COVID shock (validation fold 2020)

In [ ]:
h = H[-1]
win = fc_val[(fc_val.horizon == h) & (fc_val.fold == "val_2020") & fc_val.model.isin(["GJR", "HAR_X", "XGB"])]
piv = win.pivot(index="date", columns="model", values="forecast")
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(piv.index, np.sqrt(targets.loc[piv.index, f"rv_var_{h}"]), color="k", lw=1.5, label=f"realised vol, next {h}d")
for m, c in [("GJR", "tab:blue"), ("HAR_X", "tab:green"), ("XGB", "tab:red")]:
    ax.plot(piv.index, np.sqrt(piv[m]), lw=1, color=c, label=m)
q = fc_val[(fc_val.horizon == h) & (fc_val.fold == "val_2020") & fc_val.model.str.startswith("QGB_")]
qp = q.pivot(index="date", columns="model", values="forecast")
ax.fill_between(qp.index, np.sqrt(qp["QGB_q10"]), np.sqrt(qp["QGB_q90"]), color="tab:red", alpha=0.12, label="QGB 10–90%")
ax.set_ylabel("annualised vol"); ax.legend(ncol=3, fontsize=8)
ax.set_title(f"{h}-day volatility forecasts vs what happened, 2020 (validation)")
savefig(fig, "covid_forecasts"); plt.show()

## 7. Test-period forecasts (sealed window, 2023–2025)

The frozen configuration is reloaded from disk and its hash checked. Forecasts are generated with exactly the same function as in validation:
- annual refits for HAR and the ML models
- monthly refits for GARCH

**No loss is computed here.** Notebook 4 scores the test set once, with significance tests.

In [ ]:
text_on_disk = (OUT / "frozen_config.yaml").read_text()
assert hashlib.sha256(text_on_disk.encode()).hexdigest() == FROZEN_HASH, "frozen config changed after freezing"
frozen_loaded = yaml.safe_load(text_on_disk)

fc_test, gp_test = forecast_folds(test_folds, X, targets, panel, feature_sets, frozen_loaded, log=log)
log(f"test forecasts: {len(fc_test):,} rows")
fc = pd.concat([fc_val, fc_test], ignore_index=True)
fc.groupby(["split", "horizon"]).agg(models=("model", "nunique"), rows=("forecast", "size"),
                                     first=("date", "min"), last=("date", "max"))

## 8. Regime labels (thresholds from each training window)

Each evaluation day is labelled low, mid or high volatility using trailing 20-day realised variance. The 33rd and 67th percentiles are computed on **that fold's training data only**. Notebook 4 reports losses by regime.

In [ ]:
trailing = X["lv_cc_20"]
reg_val, thr_val = regimes_for_folds(val_folds[H[-1]], trailing, cfg["regimes"]["quantiles"])
reg_test, thr_test = regimes_for_folds(test_folds[H[-1]], trailing, cfg["regimes"]["quantiles"])
regimes = pd.concat([reg_val, reg_test], ignore_index=True)
thresholds = pd.concat([thr_val, thr_test])
display(thresholds[["low_upper_vol", "mid_upper_vol"]].rename(columns=lambda c: c + " (ann.)"))
regimes.groupby(["split", "regime"]).size().unstack()

## 9. GARCH parameter stability (monthly refits)

If the GARCH parameters drift sharply between refits, the model is unstable, and a forecast error may reflect estimation noise rather than misspecification. The plot shows each parameter path across the monthly expanding-window refits.

In [ ]:
garch_params = pd.concat([gp_val, gp_test]).sort_index()
param_paths = {spec: g for spec, g in garch_params.groupby("model")}
garch_params.to_parquet(OUT / "garch_params.parquet")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, spec in zip(axes, M.GARCH_SPECS):
    p = param_paths[spec]
    for c in [c for c in p.columns if c in ("alpha[1]", "beta[1]", "gamma[1]") and p[c].notna().any()]:
        ax.plot(p.index, p[c], label=c)
    ax.axvline(pd.Timestamp(cfg["splits"]["test_start"]), color="tab:purple", ls=":", lw=1)
    ax.set_title(f"{spec}: converged {p.converged.mean():.0%} of refits", fontsize=10); ax.legend(fontsize=8)
fig.tight_layout(); savefig(fig, "garch_param_paths"); plt.show()

## 10. Save outputs

In [ ]:
fc.to_parquet(OUT / "oos_forecasts.parquet")
regimes.to_parquet(OUT / "regimes.parquet")
thresholds.to_csv(RES / "regime_thresholds.csv")
outputs = {n: acquire.sha256(OUT / n) for n in
           ["oos_forecasts.parquet", "frozen_config.yaml", "regimes.parquet", "garch_params.parquet"]}
meta3 = {"notebook": "03_models_walkforward", "data_mode": DATA_MODE, "env": env,
         "frozen_config_sha256": FROZEN_HASH, "outputs_sha256": outputs,
         "inputs_sha256": {"features.parquet": meta2["outputs_sha256"]["features.parquet"],
                           "targets.parquet": meta1["outputs_sha256"]["targets.parquet"]},
         "total_optuna_trials": int(len(trials)), "runtime_min": round((time.time() - T0) / 60, 1)}
(OUT / "nb03_meta.json").write_text(json.dumps(meta3, indent=2, default=str))
log("saved")
pd.Series(outputs, name="sha256").to_frame()

## 11. Findings carried forward

In [ ]:
b = board.copy()
findings = {
    "total_optuna_trials": int(len(trials)),
    "best_val_QLIKE_h5": b[f"QLIKE h={H[0]}"].idxmin(),
    "best_val_QLIKE_h20": b[f"QLIKE h={H[-1]}"].idxmin(),
    "val_QLIKE_h20": {m: round(float(b.loc[m, f"QLIKE h={H[-1]}"]), 4) for m in ["GARCH", "GJR", "HAR", "HAR_X", "RF", "XGB"]},
    "garch_best_BIC_dev": diag["BIC"].idxmin(),
    "gjr_gamma_tstat_dev": round(float(diag.loc["GJR", "gamma_tstat"]), 2),
    "student_t_nu_dev": round(float(diag.loc["GJR", "nu (t dof)"]), 2),
    "frozen_config_sha256": FROZEN_HASH[:16],
    "runtime_min": round((time.time() - T0) / 60, 1),
}
(OUT / "nb03_findings.json").write_text(json.dumps(findings, indent=2))
pd.Series(findings, name="value").to_frame()

**Next: Notebook 4** scores the sealed test set once. It reports:
- QLIKE, MSE, RMSE and R²<sub>OOS</sub> against HAR-X and EWMA
- Diebold-Mariano tests with HAC variance at lag *h* − 1, and the Model Confidence Set
- block-bootstrap confidence intervals
- regime and stress-period breakdowns
- the feature ablation (E1 → E4, no-VIX)
- calibration of the 10–90% intervals